# 01 — Explore the FTW catalogs

**Purpose:** work out which FTW files we need, and which country/year to use, before downloading anything.

- **FTW-v2 benchmark-data**: one collection per country and year. Each one holds field polygons with crop labels, chips (2 km squares) carrying the spatially blocked train/val/test split, and precomputed per-field AlphaEarth (AEF) and OlmoEarth embedding stats: the mean and std of 64 dimensions over each field's 10 m pixels.
- **harmonized-field-data**: the source parcel declarations that the benchmark is cut from, in the fiboa schema, with one edition per year. Checked here only to confirm that the benchmark already carries the same fields and labels.
- **Not used:** `global-data-2e`. It holds FTW *model-predicted* boundaries, with no crop labels and no embeddings.

Everything here reads JSON metadata and Parquet footers or single columns. No geometry or embedding values are loaded.

**Outputs:** `data/raw/benchmark_tree.txt`, `data/raw/harmonized_tree.txt`, and the choice of collection (`COLLECTION`) for notebook 02.

In [2]:
import json
import fsspec
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from functools import partial
from urllib.parse import urljoin
from urllib.request import urlopen
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor

In [3]:
BENCHROOT  = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
HARMROOT   = 'https://data.source.coop/ftw/harmonized-field-data/catalog.json'
RAWDIR     = Path('../data/raw')
COLLECTION = 'ee/ee_2024'
YEAR       = 2024
MINFIELDS  = 100
NWORKERS   = 16
FS         = fsspec.filesystem('https')
RAWDIR.mkdir(parents=True,exist_ok=True)
pd.set_option('display.max_colwidth',90)
pd.set_option('display.max_rows',200)

In [4]:
def get_json(url):
    with urlopen(url,timeout=60) as response:
        return json.load(response)

def read_node(url,depth):
    node  = get_json(url)
    links = node.get('links',[])
    return dict(
        depth=depth,
        id=node['id'],
        type=node['type'],
        title=node.get('title'),
        url=url,
        assets={key:urljoin(url,asset['href']) for key,asset in node.get('assets',{}).items()},
        children=[urljoin(url,link['href']) for link in links if link['rel']=='child' and urljoin(url,link['href']).startswith(url.rsplit('/',1)[0])],
        editions={link.get('title',link['href']):urljoin(url,link['href']) for link in links if link['rel']=='item'},
        tablecolumns=[column['name'] for column in node.get('table:columns',[])])

def walk_catalog(rooturl,maxdepth=2):
    nodes = [read_node(rooturl,0)]
    with ThreadPoolExecutor(NWORKERS) as executor:
        for depth in range(1,maxdepth+1):
            urls   = [url for node in nodes if node['depth']==depth-1 for url in node['children']]
            nodes += list(executor.map(partial(read_node,depth=depth),urls))
    return sorted(nodes,key=lambda node:node['url'].rsplit('/',1)[0])

def write_tree(nodes,filepath,maxdepth=2):
    lines = []
    for node in nodes:
        indent = '    '*node['depth']
        lines.append(f"{indent}{node['id']}  [{node['type']}]  {node['title']}")
        lines += [f'{indent}  - asset {key}: {href}' for key,href in node['assets'].items()]
        lines += [f'{indent}  - edition {title}: {href}' for title,href in node['editions'].items()]
        if node['depth']==maxdepth and node['children']:
            lines.append(f"{indent}  - {len(node['children'])} chip catalogs (not walked)")
    filepath.write_text('\n'.join(lines)+'\n')
    return len(lines)

def parquet_meta(url):
    with FS.open(url,block_size=2**20) as f:
        pfile = pq.ParquetFile(f)
        return pfile.schema_arrow,pfile.metadata.num_rows

def schema_table(url,descriptions=None):
    schema,nrows = parquet_meta(url)
    descriptions = descriptions or {}
    table = pd.DataFrame({'column':schema.names,
                          'type':[str(field.type).replace('element: ','').split('<xmin')[0] for field in schema],
                          'description':[descriptions.get(name,'').split(' ([spec]')[0] for name in schema.names]})
    return table,nrows

def read_columns(url,columns):
    with FS.open(url,block_size=2**22) as f:
        return pq.read_table(f,columns=columns).to_pandas()

def parquet_crs(url):
    schema,_ = parquet_meta(url)
    geo = json.loads(schema.metadata.get(b'geo',b'{}'))
    crs = geo.get('columns',{}).get('geometry',{}).get('crs')
    if crs is None:
        return 'EPSG:4326 (GeoParquet default)' if geo else 'no geo metadata'
    return f"{crs['id']['authority']}:{crs['id']['code']}" if isinstance(crs,dict) and 'id' in crs else str(crs)[:40]

## 1. Catalog trees

Walk each catalog two levels down: root → source (country/region) → collection. The STAC tree is organised differently in each catalog:

- **benchmark-data**: root → source catalog (e.g. `ee`) → one collection per year (e.g. `ee_2024`). Each collection's children are chip catalogs, which are *not* walked; `items.parquet` lists the chips instead.
- **harmonized-field-data**: root → one collection per source (e.g. `ee`). Its yearly **editions** are STAC items (`year=2024/ee-2024.json`). They're listed but not opened.

In [5]:
trees = {'benchmark':walk_catalog(BENCHROOT),'harmonized':walk_catalog(HARMROOT)}
for name,nodes in trees.items():
    nlines = write_tree(nodes,RAWDIR/f'{name}_tree.txt')
    counts = ', '.join(f"depth {depth}: {sum(node['depth']==depth for node in nodes)}" for depth in (0,1,2))
    print(f"{name:<11} {counts}  ->  {RAWDIR/f'{name}_tree.txt'} ({nlines:,} lines)")

benchmark   depth 0: 1, depth 1: 37, depth 2: 180  ->  ../data/raw/benchmark_tree.txt (2,835 lines)
harmonized  depth 0: 1, depth 1: 79, depth 2: 0  ->  ../data/raw/harmonized_tree.txt (915 lines)


**Benchmark summary.** One row per source, listing its yearly collections and how many of them carry the `field_embeddings` asset.

In [6]:
bench = pd.DataFrame([node for node in trees['benchmark'] if node['depth']==2])
bench['source']   = bench.url.str.split('/').str[-3]
bench['year']     = bench.id.str[-4:].astype(int)
bench['hasembed'] = bench.assets.map(lambda assets:'field_embeddings' in assets)
benchsummary = (bench.groupby('source')
                .agg(collections=('id','size'),withembeddings=('hasembed','sum'),
                     years=('year',lambda years:f'{years.min()}–{years.max()}' if years.nunique()>1 else str(years.min())))
                .reset_index())
print(f'{len(bench)} benchmark collections from {bench.source.nunique()} sources; {bench.hasembed.sum()} have field_embeddings')
print('Asset keys present in every collection:',sorted(set.intersection(*[set(assets) for assets in bench.assets])))
display(benchsummary)

180 benchmark collections from 37 sources; 180 have field_embeddings
Asset keys present in every collection: ['boundary_lines', 'chips', 'chips_tiles', 'field_embeddings', 'fields', 'fields_tiles', 'items', 'style-field-coverage', 'style-outline', 'style-split', 'thumbnail']


,source,collections,withembeddings,years
0,ai4sf_ml,1,1,2021
1,at,8,8,2018–2025
2,be_vlg,8,8,2018–2025
3,be_wal,1,1,2022
4,bg,5,5,2021–2025
5,br_ba_lem,1,1,2020
6,ch,18,18,2025
7,cz,7,7,2019–2025
8,de_nds,1,1,2025
9,de_sh,3,3,2023–2025


**Harmonized summary.** One row per source collection. `croplabels` means its column list includes `crop:code` or `hcat:code`. Sources without crop labels are mostly boundary-only datasets, such as the African and Asian ML sets. `hcatstyle` means the collection has a `style-hcat-crops` map style, which only exists when the national codes were mapped to HCAT. The two counts differ for sources that have national `crop:code` but no `hcat:code` (mostly German field blocks and two Spanish regions).

In [7]:
harm = pd.DataFrame([node for node in trees['harmonized'] if node['depth']==1])
harm['years']      = harm.editions.map(lambda editions:', '.join(title.split('— ')[-1] for title in editions))
harm['editions']   = harm.editions.map(len)
harm['croplabels'] = harm.tablecolumns.map(lambda columns:bool({'crop:code','hcat:code'}&set(columns)))
harm['hcatstyle']  = harm.assets.map(lambda assets:'style-hcat-crops' in assets)
differ = harm[harm.croplabels!=harm.hcatstyle]
print(f"{len(harm)} harmonized collections; {harm.croplabels.sum()} list crop columns, {harm.hcatstyle.sum()} have a style-hcat-crops asset; {harm.editions.sum()} yearly editions in total")
print(f"{len(differ)} list crop columns but have no HCAT style: {', '.join(differ.id)}")
display(harm[['id','title','croplabels','hcatstyle','editions','years']].sort_values(['croplabels','hcatstyle','id'],ascending=[False,False,True]).reset_index(drop=True))

79 harmonized collections; 58 list crop columns, 48 have a style-hcat-crops asset; 399 yearly editions in total
10 list crop columns but have no HCAT style: bg, de_bw, de_by_block, de_he, de_mv, de_sl_block, de_st, es_an, es_ib, it_bz


,id,title,croplabels,hcatstyle,editions,years
0,at,Field boundaries for Austria,True,True,11,"2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025"
1,be_vlg,"Field boundaries for Flanders, Belgium",True,True,19,"2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 20..."
2,be_wal,Belgium Wallonia: Parcellaire Agricole Anonyme,True,True,9,"2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023"
3,ch,Field boundaries for Switzerland,True,True,40,"2017, 2017, 2018, 2018, 2019, 2019, 2020, 2020, 2021, 2021, 2022, 2022, 2022, 2023, 20..."
4,cz,Field boundaries for Czech,True,True,8,"2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026"
5,de_bb,"Field boundaries for Berlin / Brandenburg, Germany",True,True,15,"2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2026"
6,de_by,"Field boundaries for Bavaria, Germany",True,True,1,2026
7,de_nds,"Crop Fields for Lower Saxony / Bremen / Hamburg, Germany",True,True,1,2025
8,de_nrw,"Field blocks for North Rhine-Westphalia (NRW), Germany",True,True,6,"2019, 2020, 2021, 2022, 2023, 2026"
9,de_thuenen,"Crop type map of Germany from Sentinel-1, Sentinel-2 and Landsat (Thünen Institute)",True,True,5,"2017, 2018, 2019, 2020, 2021"


## 2. Which assets hold the embedding stats and the crop labels

These are the schemas for the probe collection (`COLLECTION`), with the column descriptions taken from the collection's own metadata (`table:columns`). The harmonized file shown is the **exact edition** the benchmark was built from, which the collection records in `ftw:config.source.href`.

In [8]:
SOURCE,NAME = COLLECTION.split('/')
collection  = get_json(f'https://data.source.coop/ftw/benchmark-data/{COLLECTION}/collection.json')
harmurl     = collection['ftw:config']['source']['href']
harmmeta    = get_json(collection['ftw:config']['source']['via'])
ASSETURLS   = {key:urljoin(f'https://data.source.coop/ftw/benchmark-data/{COLLECTION}/',collection['assets'][key]['href'])
               for key in ('fields','field_embeddings','chips')}
ASSETURLS['harmonized'] = harmurl
DESCRIPTIONS = {key:{column['name']:column.get('description','') for column in collection['assets'][key].get('table:columns',[])}
                for key in ('fields','field_embeddings','chips')}
DESCRIPTIONS['harmonized'] = {column['name']:column.get('description','') for column in harmmeta['table:columns']}
for key,url in ASSETURLS.items():
    table,nrows = schema_table(url,DESCRIPTIONS[key])
    print(f'\n{key}: {nrows:,} rows, {FS.size(url)/1e6:,.0f} MB\n{url}')
    display(table)


fields: 166,500 rows, 278 MB
https://data.source.coop/ftw/benchmark-data/ee/ee_2024/ee_2024_fields.parquet


,column,type,description
0,id,string,An identifier for the field.
1,land_use,string,Carried over from the source column `taotletud_maakasutus`; the publisher documents no...
2,crop:name,string,Crop name in the original language.
3,hcat:name_en,string,The original crop name translated into English.
4,crop:code,string,"The crop code, from the code list of the source."
5,hcat:code,uint32,"The 10-digit HCAT code indicating the hierarchy of the crop. The first 4, 6, 8 digits ..."
6,hcat:name,string,The machine-readable HCAT name of the crop (Hierarchical Crop and Agriculture Taxonomy...
7,metrics:area,float,"Area of the field, in square meters (m²). Must be > 0 and <= 1,000,000,000."
8,parcel_id,int64,"Field identifier (source column `pollu_id`, per the fiboa data survey)"
9,geometry,binary,"A geometry that reflects the footprint of the field, usually a Polygon. Stored in the ..."



field_embeddings: 166,500 rows, 154 MB
https://data.source.coop/ftw/benchmark-data/ee/ee_2024/ee_2024_field_embeddings.parquet


,column,type,description
0,id,string,Field id; joins onto the `fields` (or `fields_filtered`) table.
1,pixels,int32,Number of 10 m pixels used for this field's embedding statistics.
2,pixel_rule,string,How pixels were selected for this field: `centroid` (pixels whose centre falls inside ...
3,aef_pixels,int32,Number of pixels the AlphaEarth Foundations product covered for this field (can differ...
4,aef_mean,list<float>,Mean of each AlphaEarth Foundations Satellite Embedding dimension (64 values) over the...
5,aef_std,list<float>,Population standard deviation of each AlphaEarth Foundations Satellite Embedding dimen...
6,olmoearth_pixels,int32,Number of pixels the OlmoEarth v1.3 product covered for this field (can differ from `p...
7,olmoearth_mean,list<float>,Mean of each OlmoEarth v1.3 embedding dimension (the first 64 of 128 dimensions) over ...
8,olmoearth_std,list<float>,Population standard deviation of each OlmoEarth v1.3 embedding dimension (first 64 of ...



chips: 7,562 rows, 1 MB
https://data.source.coop/ftw/benchmark-data/ee/ee_2024/ee_2024_chips.parquet


,column,type,description
0,gzd,string,MGRS grid zone designator of the chip's grid cell
1,mgrs_10km,string,MGRS 100 km square plus 10 km cell identifier the chip belongs to
2,id,string,chip identifier; matches the STAC item id (with the year suffix stripped)
3,field_coverage_pct,double,percent of the chip's area covered by mapped field polygons
4,hcat_dominant_code,int64,EuroCrops HCAT code of the crop covering most of the chip's fields
5,hcat_dominant_name_en,string,English name for hcat_dominant_code
6,hcat_dominant_pct,double,share of the chip's field area under the dominant crop
7,hcat_top,"list<struct<code: int64, name_en: string, pct: double>>","the crops covering the chip's field area, as {code, name_en, pct} entries ordered by s..."
8,split,string,which benchmark split the chip belongs to (train / val / test)
9,admin0_iso3,string,"ISO 3166-1 alpha-3 code of the country: the unit containing the chip's centre, or the ..."



harmonized: 166,500 rows, 264 MB
https://data.source.coop/ftw/harmonized-field-data/ee/year=2024/ee-2024.parquet


,column,type,description
0,id,string,An identifier for the field.
1,land_use,string,Carried over from the source column `taotletud_maakasutus`; the publisher documents no...
2,crop:name,string,Crop name in the original language.
3,geometry,binary,"A geometry that reflects the footprint of the field, usually a Polygon. Stored in the ..."
4,hcat:name_en,string,The original crop name translated into English.
5,crop:code,string,"The crop code, from the code list of the source."
6,hcat:code,uint32,"The 10-digit HCAT code indicating the hierarchy of the crop. The first 4, 6, 8 digits ..."
7,hcat:name,string,The machine-readable HCAT name of the crop (Hierarchical Crop and Agriculture Taxonomy...
8,metrics:area,float,"Area of the field, in square meters (m²). Must be > 0 and <= 1,000,000,000."
9,parcel_id,int64,"Field identifier (source column `pollu_id`, per the fiboa data survey)"


**Where things live:**

| Need | Asset | Column(s) |
|---|---|---|
| Field ID (join key) | `fields`, `field_embeddings`, harmonized | `id` |
| Crop label | `fields` (identical to harmonized) | `hcat:code` (EuroCrops HCAT, 10-digit) and `hcat:name` (HCAT class name); national code `crop:code` / `crop:name`, with `hcat:name_en` its English translation (one per national code, not the HCAT class name) |
| AEF features | `field_embeddings` | `aef_mean`, `aef_std` (64 floats each), `aef_pixels` |
| OlmoEarth features | `field_embeddings` | `olmoearth_mean`, `olmoearth_std` (first 64 of 128 dims), `olmoearth_pixels` |
| Area / geometry | `fields` | `metrics:area` (m²), `geometry` (WKB), `bbox` |
| Split | `chips` only | `split` (train/val/test, per 2 km chip) |
| Year | collection metadata | no year column; it comes from the collection id and the temporal extent |

The fields have **no split column**. Notebook 03 gives each field the split of the chip it falls in.

## 3. Join keys, year and CRS checks

These checks load only the `id` columns. They confirm that `id` is unique and shared by all three tables, that the label year matches the embedding year, and which CRS each geometry is stored in.

In [9]:
idsfields = read_columns(ASSETURLS['fields'],['id','parcel_id'])
idsembed  = read_columns(ASSETURLS['field_embeddings'],['id'])
idsharm   = read_columns(ASSETURLS['harmonized'],['id'])
fieldset,embedset,harmset = set(idsfields.id),set(idsembed.id),set(idsharm.id)
embeddesc = collection['assets']['field_embeddings']['description']
harmitem  = get_json(urljoin(harmurl,harmurl.rsplit('/',1)[-1].replace('.parquet','.json')))
checks = pd.DataFrame([
    ('fields rows / unique id',f'{len(idsfields):,} / {idsfields.id.nunique():,}'),
    ('field_embeddings rows / unique id',f'{len(idsembed):,} / {idsembed.id.nunique():,}'),
    ('harmonized rows / unique id',f'{len(idsharm):,} / {idsharm.id.nunique():,}'),
    ('fields ∩ field_embeddings (id)',f'{len(fieldset&embedset):,}'),
    ('fields ∩ harmonized (id)',f'{len(fieldset&harmset):,}'),
    ('parcel_id unique in fields',str(idsfields.parcel_id.is_unique)),
    ('benchmark build year (ftw:config)',str(collection['ftw:config']['config']['year'])),
    ('benchmark temporal extent',' to '.join(collection['extent']['temporal']['interval'][0])),
    ('embeddings year (asset description)',next((word.strip(';,.') for word in embeddesc.split() if word.strip(';,.').isdigit()),'not stated')),
    ('harmonized edition year',str(harmitem['properties'].get('year'))),
    ('fields geometry CRS',parquet_crs(ASSETURLS['fields'])),
    ('chips geometry CRS',parquet_crs(ASSETURLS['chips'])),
    ('benchmark version',collection.get('version'))],columns=['check','value'])
display(checks)

,check,value
0,fields rows / unique id,"166,500 / 166,500"
1,field_embeddings rows / unique id,"166,500 / 166,500"
2,harmonized rows / unique id,"166,500 / 166,500"
3,fields ∩ field_embeddings (id),"166,500"
4,fields ∩ harmonized (id),"166,500"
5,parcel_id unique in fields,True
6,benchmark build year (ftw:config),2024
7,benchmark temporal extent,2024-01-01T00:00:00Z to 2024-12-31T23:59:59Z
8,embeddings year (asset description),2024
9,harmonized edition year,2024


**Takeaways:**
- `id` is the join key. Fields, embeddings and the harmonized edition hold the same ids, one row each.
- Labels and embeddings refer to the same year: the benchmark build year, the embeddings year and the harmonized edition year all agree.
- Field geometry is already in **EPSG:4326**, like the chips, even though the column description says "source CRS" (the bboxes are lon/lat and the build config has `skip_reproject=False`). No reprojection is needed for centroids or the chip join.
- The harmonized file adds nothing beyond `fields`, so the pipeline reads labels from the benchmark `fields` asset.

## 4. Choosing a country/year

First, which benchmark collections carry crop labels at all, and how big are they? Only the Parquet footers are read for this. For CPU-only work, a collection with roughly 100k–1M fields is a good size: enough fields per crop, but still quick to train on.

In [10]:
def collection_size(row):
    base = row.url.rsplit('/',1)[0]+'/'
    schema,nrows = parquet_meta(base+f'{row.id}_fields.parquet')
    return dict(id=row.id,source=row.source,year=row.year,fields=nrows,hcat='hcat:code' in schema.names,
                fieldsmb=FS.size(base+f'{row.id}_fields.parquet')/1e6,embedmb=FS.size(base+f'{row.id}_field_embeddings.parquet')/1e6)

with ThreadPoolExecutor(NWORKERS) as executor:
    sizes = pd.DataFrame(list(executor.map(collection_size,bench.itertuples())))
labelled = sizes[sizes.hcat]
latest   = labelled.sort_values('year').groupby('source').tail(1).sort_values('fields').reset_index(drop=True)
print(f'{len(labelled)}/{len(sizes)} benchmark collections carry HCAT crop labels')
print(f"Without labels: {', '.join(sorted(sizes.loc[~sizes.hcat,'source'].unique()))}")
print('\nLatest labelled year per source:')
display(latest.style.format({'fields':'{:,}','fieldsmb':'{:,.0f}','embedmb':'{:,.0f}'}).hide(axis='index'))

147/180 benchmark collections carry HCAT crop labels
Without labels: ai4sf_ml, bg, br_ba_lem, de_sh, es_ib, glocab, in_10k_ml, jp, lacuna, rw_rwanda_ml, wri_trazo_ml

Latest labelled year per source:


id,source,year,fields,hcat,fieldsmb,embedmb
za_fusion_ml_2017,za_fusion_ml,2017,"6,568",True,2,6
ch_ag_2025,ch,2025,"97,527",True,28,90
ee_2024,ee,2024,"166,500",True,278,154
sk_2025,sk,2025,"273,387",True,497,254
be_wal_2022,be_wal,2022,"341,968",True,109,317
cz_2025,cz,2025,"411,105",True,338,381
lv_2025,lv,2025,"417,383",True,628,387
ec_lt_2021,ec_lt,2021,"439,199",True,218,407
be_vlg_2025,be_vlg,2025,"594,732",True,175,542
dk_2025,dk,2025,"616,411",True,471,572


**Choice: `ee/ee_2024` (Estonia 2024).**
- It has 166,500 fields with complete HCAT labels. That's large enough for dozens of crops with ≥100 fields, and small enough (~430 MB for fields + embeddings) to train comfortably on CPU.
- 2024 is covered by both AEF and OlmoEarth for every field (checked below).
- The chips carry a spatially blocked split (`block3x3`: 3×3 blocks of 2 km chips, so 6 km blocks, 80/10/10), so test fields can be kept geographically separate from training fields.

To use a different country, change `COLLECTION` and rerun. Belgium Flanders 2024 (`be_vlg`, ~600k fields) is a larger alternative with a more arable crop mix.

**Label completeness and pixel coverage** for the chosen collection. Only the label and pixel-count columns are loaded; the embedding lists are not.

In [11]:
labels = read_columns(ASSETURLS['fields'],['id','hcat:code','hcat:name_en','hcat:name','crop:code','crop:name','metrics:area'])
pixels = read_columns(ASSETURLS['field_embeddings'],['id','pixels','pixel_rule','aef_pixels','olmoearth_pixels'])
fields = labels.merge(pixels,on='id',how='left',validate='one_to_one')
nfields = len(fields)
summary = pd.DataFrame([
    ('fields',f'{nfields:,}'),
    ('missing hcat:code',f"{fields['hcat:code'].isna().sum():,} ({fields['hcat:code'].isna().mean():.2%})"),
    ('missing crop:code',f"{fields['crop:code'].isna().sum():,} ({fields['crop:code'].isna().mean():.2%})"),
    ('distinct HCAT classes',f"{fields['hcat:code'].nunique():,}"),
    ('distinct national crop codes',f"{fields['crop:code'].nunique():,}"),
    ('no AEF pixels',f"{(fields.aef_pixels.fillna(0)==0).sum():,}"),
    ('no OlmoEarth pixels',f"{(fields.olmoearth_pixels.fillna(0)==0).sum():,}"),
    ('AEF pixels < total pixels',f'{(fields.aef_pixels<fields.pixels).sum():,}'),
    ("pixel_rule = 'touched' (too small for a pixel centre)",f"{(fields.pixel_rule=='touched').sum():,}"),
    ('fields with < 5 pixels',f'{(fields.pixels<5).sum():,} ({(fields.pixels<5).mean():.2%})'),
    ('area median / min / max (ha)',f"{fields['metrics:area'].median()/1e4:.2f} / {fields['metrics:area'].min()/1e4:.2f} / {fields['metrics:area'].max()/1e4:,.1f}")],
    columns=['metric','value'])
display(summary.style.hide(axis='index'))

metric,value
fields,"166,500"
missing hcat:code,0 (0.00%)
missing crop:code,0 (0.00%)
distinct HCAT classes,123
distinct national crop codes,251
no AEF pixels,0
no OlmoEarth pixels,0
AEF pixels < total pixels,69
pixel_rule = 'touched' (too small for a pixel centre),234
fields with < 5 pixels,"2,946 (1.77%)"


**Crop-code lookup and fields per crop.** HCAT is the EuroCrops Hierarchical Crop and Agriculture Taxonomy. It's a 10-digit code where the first 4, 6 and 8 digits select increasingly specific crop groups (`hcat:code` description in the collection metadata). `hcat:name` is the HCAT class name and is the label we use. Each national code (`crop:code`) was mapped to one HCAT class by the fiboa converter; `hcat:name_en` is just the English translation of the national name, so several of them can share one HCAT class. `natcodes` counts how many national codes merge into each HCAT class, and `examplenational` shows one of them. Classes below `MINFIELDS` are flagged; notebook 02 drops them.

In [12]:
lookup = (fields.groupby(['hcat:code','hcat:name'])
          .agg(nfields=('id','size'),natcodes=('crop:code','nunique'),examplenational=('hcat:name_en','first'))
          .reset_index().sort_values('nfields',ascending=False).reset_index(drop=True))
lookup['pct']   = 100*lookup.nfields/nfields
lookup['keep']  = lookup.nfields>=MINFIELDS
lookup['hcat6'] = lookup['hcat:code'].astype(str).str[:6]
kept  = lookup[lookup.keep]
share = kept.nfields.sum()/nfields
print(f'{len(kept)} of {len(lookup)} HCAT classes have ≥ {MINFIELDS} fields, covering {kept.nfields.sum():,} fields ({share:.1%})')
print(f"Largest class: {lookup.loc[0,'hcat:name']} ({lookup.loc[0,'pct']:.1f}% of fields); smallest kept: {kept.iloc[-1]['hcat:name']} ({kept.iloc[-1]['nfields']:,} fields)")
display(lookup.style.format({'nfields':'{:,}','pct':'{:.2f}'}).hide(axis='index'))

37 of 123 HCAT classes have ≥ 100 fields, covering 164,880 fields (99.0%)
Largest class: pasture_meadow_grassland_grass (44.9% of fields); smallest kept: blueberry (104 fields)


hcat:code,hcat:name,nfields,natcodes,examplenational,pct,keep,hcat6
3302000000,pasture_meadow_grassland_grass,"74,806",3,grasses,44.93,True,330200
3301010101,winter_common_soft_wheat,"14,180",3,Winter wheat,8.52,True,330101
3301090300,legumes_harvested_green,"12,489",7,mixture of legumes and grasses (30-80% legumes),7.50,True,330109
3301010500,oats,"7,680",3,Oats undersown with legumes,4.61,True,330101
3301010402,spring_barley,"7,106",3,Summer barley undersown with legumes,4.27,True,330101
3301090303,clover,"6,788",15,red clover (clover 50-80% other grasses 20-50%),4.08,True,330109
3301020600,peas,"6,159",9,Other field pea (100%),3.70,True,330102
3301060401,winter_rapeseed_rape,"4,624",6,Winter oilseed rape,2.78,True,330106
3301010102,spring_common_soft_wheat,"4,566",3,Summer wheat,2.74,True,330101
3301030000,potatoes,"2,883",6,Other potato,1.73,True,330103


**Split coverage (chips).** The split lives on the chips. These are the chip counts and the field coverage per split, a first sanity check before fields are assigned to chips in notebook 03.

In [13]:
chips = read_columns(ASSETURLS['chips'],['id','split','field_coverage_pct','mgrs_10km','admin1_name'])
splits = (chips.groupby('split')
          .agg(chips=('id','size'),mgrs10km=('mgrs_10km','nunique'),counties=('admin1_name','nunique'),
               mediancoverage=('field_coverage_pct','median'))
          .reindex(['train','val','test']))
splits['pctchips'] = 100*splits.chips/splits.chips.sum()
print(f"split type: {collection['ftw:split_type']}, percents: {collection['ftw:split_percents']}, seed: {collection['ftw:split_seed']}")
display(splits.style.format({'chips':'{:,}','mediancoverage':'{:.1f}','pctchips':'{:.1f}'}))

split type: block3x3, percents: [80, 10, 10], seed: 42


,chips,mgrs10km,counties,mediancoverage,pctchips
split,,,,,
train,"6,037",432,16,24.5,79.8
val,733,153,15,21.9,9.7
test,792,153,14,28.9,10.5


## 5. Label check across all labelled 2024 collections

Notebook 02 loops over every collection, so this checks each labelled `YEAR` collection up front: the number of fields, the % with an HCAT label, and how many HCAT classes reach `MINFIELDS`. Only the `hcat:code` column is read from each file.

In [ ]:
def label_check(row):
    url    = bench.set_index('id').loc[row.id,'url'].rsplit('/',1)[0]+f'/{row.id}_fields.parquet'
    codes  = read_columns(url,['hcat:code'])['hcat:code']
    counts = codes.value_counts()
    kept   = counts[counts>=MINFIELDS]
    return dict(id=row.id,fields=len(codes),pctlabelled=100*codes.notna().mean(),classes=len(counts),
                keptclasses=len(kept),pctkept=100*kept.sum()/len(codes),embedmb=row.embedmb)

europe = labelled[labelled.year==YEAR]
with ThreadPoolExecutor(NWORKERS) as executor:
    checks = pd.DataFrame(list(executor.map(label_check,europe.itertuples()))).sort_values('fields').reset_index(drop=True)
print(f'{len(checks)} labelled {YEAR} collections hold {checks.fields.sum():,} fields and {checks.embedmb.sum()/1e3:,.1f} GB of embeddings')
print(f'{(checks.pctlabelled<100).sum()} have missing labels; {(checks.keptclasses<10).sum()} have fewer than 10 classes with ≥ {MINFIELDS} fields')
display(checks.style.format({'fields':'{:,}','pctlabelled':'{:.1f}','pctkept':'{:.1f}','embedmb':'{:,.0f}'}).hide(axis='index'))

## Summary for notebook 02

- **Collection:** `COLLECTION` (above). Read `fields` (labels, area, geometry) and `field_embeddings` (AEF/OlmoEarth mean/std). Join on `id`.
- **Labels:** `hcat:code` / `hcat:name` (the HCAT class). `hcat:name_en` is the national crop name in English, not the class name. Drop classes with fewer than `MINFIELDS` fields (flagged in the lookup table).
- **Decision for notebook 02:** `fallow_land_not_crop` (1,851 fields in Estonia) is a non-crop class. Decide whether to keep it as a class or drop it.
- **Small fields:** drop fields with very few pixels, including `pixel_rule == 'touched'`.
- **Geometry:** already EPSG:4326. No reprojection is needed for centroid lon/lat or the chip join.
- **Split:** taken from the chips in notebook 03.